In [ ]:
SR = 48000

from math import exp
from random import random

def vca(wave, signal):
    if len(wave) != len(signal):
        raise ValueError("The sound and control signal must have equal lengths.")
    return [x * gain for x, gain in zip(wave, signal)]

def noise(duration=0.05, amplitude=0.15, sample_rate=SR):
    n = round(duration * sample_rate)
    return [amplitude * (2 * random() - 1) for _ in range(n)]

def envelope_lin(duration=0.05, sample_rate=SR):
    n = round(duration * sample_rate)
    denominator = max(1, n - 1)
    return [1 - i / denominator for i in range(n)]

def square_wave_f(f, sample_rate=SR, amplitude=0.15):
    phase = 0.0
    wave = []
    for frequency in f:
        if not 0 < frequency < sample_rate / 2:
            raise ValueError("Each frequency must be positive and below half the sample rate.")
        wave.append(-amplitude if phase < 0.5 else amplitude)
        phase = (phase + frequency / sample_rate) % 1
    return wave

def edge_fade(wave, attack=0.001, release=0.005, sample_rate=SR):
    n = len(wave)
    attack_samples = max(1, round(attack * sample_rate))
    release_samples = max(1, round(release * sample_rate))
    edge = [
        min(1.0, i / attack_samples, (n - 1 - i) / release_samples)
        for i in range(n)
    ]
    return vca(wave, edge)

def envelope_exp(duration=0.05, tau=0.012, sample_rate=SR):
    if tau <= 0:
        raise ValueError("tau must be positive.")
    n = round(duration * sample_rate)
    return [exp(-(i / sample_rate) / tau) for i in range(n)]

def closed_hat(duration=0.06, tau=0.010, amplitude=0.15, sample_rate=SR):
    source = noise(duration, amplitude=amplitude, sample_rate=sample_rate)
    shape = envelope_exp(duration, tau=tau, sample_rate=sample_rate)
    return edge_fade(vca(source, shape), sample_rate=sample_rate)

def open_hat(duration=0.2, tau=0.070, amplitude=0.15, sample_rate=SR):
    source = noise(duration, amplitude=amplitude, sample_rate=sample_rate)
    shape = envelope_exp(duration, tau=tau, sample_rate=sample_rate)
    return edge_fade(vca(source, shape), sample_rate=sample_rate)

def kick_raw(
    duration=0.25, start_hz=180, end_hz=45,
    pitch_tau=0.025, amp_tau=0.060,
    amplitude=0.20, sample_rate=SR,
):
    pitch_shape = envelope_exp(duration, tau=pitch_tau, sample_rate=sample_rate)
    pitch = [end_hz + (start_hz - end_hz) * value for value in pitch_shape]
    body = square_wave_f(pitch, sample_rate=sample_rate, amplitude=amplitude)
    shape = envelope_exp(duration, tau=amp_tau, sample_rate=sample_rate)
    return vca(body, shape)

def kick(
    duration=0.25, start_hz=180, end_hz=45,
    pitch_tau=0.025, amp_tau=0.060,
    amplitude=0.20, sample_rate=SR,
):
    wave = kick_raw(
        duration=duration, start_hz=start_hz, end_hz=end_hz,
        pitch_tau=pitch_tau, amp_tau=amp_tau,
        amplitude=amplitude, sample_rate=sample_rate,
    )
    return edge_fade(wave, sample_rate=sample_rate)

def mix(a, b, ga=1.0, gb=1.0):
    if len(a) != len(b):
        raise ValueError("Pad the shorter sound with silence before mixing.")
    return [ga * x + gb * y for x, y in zip(a, b)]

def snare(
    duration=0.20, amplitude=0.1, noise_gain=0.5,
    sample_rate=SR,
):
    pitch = [
        180 + 120 * value
        for value in envelope_exp(duration, tau=0.010, sample_rate=sample_rate)
    ]
    body = vca(
        square_wave_f(pitch, amplitude=amplitude, sample_rate=sample_rate),
        envelope_exp(duration, tau=0.025, sample_rate=sample_rate),
    )
    wires = vca(
        noise(duration, amplitude=amplitude, sample_rate=sample_rate),
        envelope_exp(duration, tau=0.055, sample_rate=sample_rate),
    )
    return edge_fade(
        mix(body, wires, ga=1.0, gb=noise_gain),
        sample_rate=sample_rate,
    )


In [ ]:
def fit(wave, n_samples):
    return wave[:n_samples] + [0] * (n_samples - len(wave))

eight_duration = 0.3
eight_samples = round(SR * eight_duration)

slot_s = round(SR/2)

In [ ]:
def mixreal(signal1, signal2):
    n = max(len(signal1), len(signal2))

    mixed = []

    for i in range(n):
        sample1 = signal1[i] if i < len(signal1) else 0
        sample2 = signal2[i] if i < len(signal2) else 0

        mixed.append(sample1 + sample2)

    return mixed

In [ ]:
from IPython.display import Audio
bombo = kick_raw()
redo = snare()
hatclosed = closed_hat()
hatopen = open_hat()
Audio(hatopen, rate=SR)



In [ ]:
T = 20
Bpm = 120
Bps = Bpm/60
negra_duration = 1/Bps
blanca_duration = negra_duration*2
corchea_duration = negra_duration/2
semicorchea_duration = negra_duration/4

negra_samples = round(negra_duration * 48000)
blanca_samples = round(blanca_duration * 48000)
corchea_samples = round(corchea_duration * 48000)
semicorchea_samples = round(semicorchea_duration * 48000)
Bombo = (bombo + [0]*(negra_samples-len(bombo)))*40
Charles = ([0]*corchea_samples + hatclosed + [0]*(corchea_samples-len(hatclosed)))*40
Percu1 = mixreal(Bombo, Charles)
Audio(Percu1, rate = SR)

In [ ]:
T = 20
Bpm = 120
Bps = Bpm/60
negra_duration = 1/Bps #0.5
blanca_duration = negra_duration*2
corchea_duration = negra_duration/2   #0.25
semicorchea_duration = negra_duration/4

negra_samples = round(negra_duration * 48000)
blanca_samples = round(blanca_duration * 48000)
corchea_samples = round(corchea_duration * 48000)
semicorchea_samples = round(semicorchea_duration * 48000)
Bombo = (bombo + [0]*(corchea_samples-len(bombo)) + bombo + [0]*(corchea_samples-len(bombo)) + [0]*negra_samples + bombo + [0]*(corchea_samples-len(bombo)) + [0]*negra_samples + [0]*corchea_samples)*10
Charles = ([0]*corchea_samples*3 + hatclosed + [0]*(corchea_samples-len(hatclosed))+ [0]*corchea_samples+ hatclosed + [0]*(corchea_samples-len(hatclosed))+ [0]*corchea_samples+ hatclosed + [0]*(corchea_samples-len(hatclosed)))*10
Redo = ([0]*negra_samples + redo + [0]*(negra_samples-len(redo)))*20
Percu1 = mixreal(Bombo, Redo)
Percu2 = mixreal(Percu1, Charles)
Audio(Percu2, rate = SR)

In [ ]:
T = 20
Bpm = 120
Bps = Bpm/60
negra_duration = 1/Bps #0.5
blanca_duration = negra_duration*2
corchea_duration = negra_duration/2   #0.25
semicorchea_duration = negra_duration/4

negra_samples = round(negra_duration * 48000)
blanca_samples = round(blanca_duration * 48000)
corchea_samples = round(corchea_duration * 48000)
semicorchea_samples = round(semicorchea_duration * 48000)
Bombo = (bombo + [0]*(negra_samples-len(bombo)))*40
Charles = ([0]*corchea_samples + hatopen + [0]*(corchea_samples-len(hatopen)))*40
Redo = ([0]*negra_samples + redo + [0]*(corchea_samples-len(redo))+ [0]*corchea_samples)*20
Percu1 = mixreal(Bombo, Redo)
Percu2 = mixreal(Percu1, Charles)
Audio(Percu2, rate = SR)

In [ ]:
T = 20
Bpm = 95
Bps = Bpm/60
negra_duration = 1/Bps #0.5
blanca_duration = negra_duration*2
corchea_duration = negra_duration/2   #0.25
semicorchea_duration = negra_duration/4

negra_samples = round(negra_duration * 48000)
blanca_samples = round(blanca_duration * 48000)
corchea_samples = round(corchea_duration * 48000)
semicorchea_samples = round(semicorchea_duration * 48000)
Bombo = (bombo + [0]*(negra_samples-len(bombo)))*40
Charles = ([0]*corchea_samples + hatclosed + [0]*(corchea_samples-len(hatclosed)))*40
Redo = ([0]*semicorchea_samples*3 + redo + [0]*(semicorchea_samples-len(redo))+ [0]*semicorchea_samples*2+[0]*(semicorchea_samples-len(redo))+[0]*semicorchea_samples)*20
Percu1 = mixreal(Bombo, Redo)
Percu2 = mixreal(Percu1, Charles)
Audio(Percu1, rate = SR)

In [ ]:
def render_pattern (pattern, instrument, BPM = 130 , rate = SR):
    fourth_s = 60/BPM
    slot_n = round(fourth_s * SR)

    beats = [fit(instrument() if x == 1 else [], slot_n) for x in parse(pattern)]
    rendered_pattern = sum(beats, [])
    return rendered_pattern


In [ ]:
def play(wave, sample_rate=SR):
    return Audio(wave, rate=sample_rate, normalize=False)

def silence(duration=0.5, sample_rate=SR):
    return [0.0] * round(duration * sample_rate)

In [ ]:
play(render_pattern("x.xx", kick_raw))

In [ ]:
def render_pattern (pattern, instrument, nth = 4, BPM = 110 , rate = SR):
    slot_s = (60/BPM) * 4 / nth
    slot_n = round(slot_s * SR)

    beats = [fit(instrument() if x == 1 else [], slot_n) for x in parse(pattern)]
    rendered_pattern = sum(beats, [])
    return rendered_pattern

In [ ]:
def multiple_mix(waves):
    result = []
    max_len = max(len(w) for w in waves) #cerco il segnale più lungo
    for w in waves:
        result = mix(fit(result, max_len), fit(w, max_len))
    return result


In [ ]:
def kick(duration=0.02, samplerate=48000, amplitude=1):

    n = round(duration * samplerate)
    kick_wave = []

    for i in range(n):

        t = i / samplerate

        # Frecuencia que baja rápidamente
        f = 150 * (1 - t / duration) + 50

        # Envolvente para que el sonido desaparezca
        envelope = 1 - t / duration

        # Oscilación
        sample = amplitude * envelope * (
            __import__('math').sin(2 * __import__('math').pi * f * t)
        )

        kick_wave.append(sample)

    return kick_wave

In [ ]:
def bass(duration=0.1, freq = 100,  samplerate=48000, amplitude=0.35):

    n = round(duration * samplerate)
    kick_wave = []

    for i in range(n):

        t = i / samplerate

        # Frecuencia que baja rápidamente
        f = freq

        # Envolvente para que el sonido desaparezca
        envelope = 1 - t / duration

        # Oscilación
        sample = amplitude * envelope * (
            __import__('math').sin(2 * __import__('math').pi * f * t)
        )

        kick_wave.append(sample)

    return kick_wave


BASS NOTES

In [ ]:
bassbb = bass(duration=0.1, freq=116.541)
bassf = bass(duration=0.1, freq=87.3)
bassg = bass(duration=0.1, freq=103.826)
bassf2 = bass(duration=0.1, freq=87.3*2)
bassg2 = bass(duration=0.1, freq=103.826*2)
bassbb2 = bass(duration=0.1, freq=116.541*2)




Audio(bassf2, rate=48000)

CREATING BASS LINE

In [ ]:
T = 20
Bpm = 110
Bps = Bpm/60
negra_duration = 1/Bps #0.5
blanca_duration = negra_duration*2
corchea_duration = negra_duration/2   #0.25
semicorchea_duration = negra_duration/4

negra_samples = round(negra_duration * 48000)
blanca_samples = round(blanca_duration * 48000)
corchea_samples = round(corchea_duration * 48000)
semicorchea_samples = round(semicorchea_duration * 48000)
Bombo = (bassbb + [0]*(semicorchea_samples-len(bassbb)) + bassg + [0]*(semicorchea_samples-len(bassbb))+bassf + [0]*(semicorchea_samples-len(bassbb))+[0]*(semicorchea_samples)*3+bassf+[0]*(semicorchea_samples-len(bassbb))+[0]*(semicorchea_samples)*3+ bassf+[0]*(semicorchea_samples-len(bassbb))+[0]*(semicorchea_samples)*6+ (bassf+[0]*(semicorchea_samples-len(bassbb)))*2+[0]*semicorchea_samples+ bassf+[0]*(semicorchea_samples-len(bassbb))+[0]*semicorchea_samples+bassg+[0]*(semicorchea_samples-len(bassbb))+[0]*semicorchea_samples+ bassf+[0]*(semicorchea_samples-len(bassbb))+ bassbb+[0]*(semicorchea_samples-len(bassbb))+[0]*semicorchea_samples*6)*2
Bass2 =(bassbb2 + [0]*(semicorchea_samples-len(bassbb)) + bassg2 + [0]*(semicorchea_samples-len(bassbb))+bassf2 + [0]*(semicorchea_samples-len(bassbb))+[0]*(semicorchea_samples)*3+bassf2+[0]*(semicorchea_samples-len(bassbb))+[0]*(semicorchea_samples)*3+ bassf2+[0]*(semicorchea_samples-len(bassbb))+[0]*(semicorchea_samples)*6+ (bassf2+[0]*(semicorchea_samples-len(bassbb)))*2+[0]*semicorchea_samples+ bassf2+[0]*(semicorchea_samples-len(bassbb))+[0]*semicorchea_samples+bassg2+[0]*(semicorchea_samples-len(bassbb))+[0]*semicorchea_samples+ bassf2+[0]*(semicorchea_samples-len(bassbb))+ bassbb2+[0]*(semicorchea_samples-len(bassbb))+[0]*semicorchea_samples*6)

Bombo2 = (bassbb + [0]*(semicorchea_samples-len(bassbb)) + bassg + [0]*(semicorchea_samples-len(bassbb))+bassf + [0]*(semicorchea_samples-len(bassbb))+[0]*(semicorchea_samples)*3+bassf+[0]*(semicorchea_samples-len(bassbb))+[0]*(semicorchea_samples)*3+ bassf+[0]*(semicorchea_samples-len(bassbb))+[0]*(semicorchea_samples)*6+ (bassf+[0]*(semicorchea_samples-len(bassbb)))*2+[0]*semicorchea_samples+ bassf+[0]*(semicorchea_samples-len(bassbb))+[0]*semicorchea_samples+bassg+[0]*(semicorchea_samples-len(bassbb))+[0]*semicorchea_samples+ bassf+[0]*(semicorchea_samples-len(bassbb))+ bassbb+[0]*(semicorchea_samples-len(bassbb))+[0]*semicorchea_samples*6)
Basss = Bombo +mix(Bass2,Bombo2)*2

Audio(Basss, rate = SR)

FINAL RESULT

In [ ]:
Queen = multiple_mix([render_pattern("..", kick, 16)])+multiple_mix([render_pattern("xxxx", kick, 4), render_pattern(".x.x", snare, 4), render_pattern("xx.xxx.x",closed_hat , 8), render_pattern("..x...x.",open_hat , 8)])*8
Audio(mixreal(Basss, Queen), rate = 48000)


PERCUSSION

In [ ]:
Audio(Queen, rate = 48000)